<h1> Associating coordinates to texas oil leases and combining it with dispensing data <h1>

This program will associate ccordinates to leases in pdq_well_cycle_disp dataset by using a lease_key. This will be relatively simple since we have alredy made a data set for lease_key and coordinates in pdq_add_coordinates.ipynb. We will make a lease key for pdq_well_cycle_disp using the oil gas code, dist_no and lease_no and merge them appropriately.

In [25]:
from pathlib import Path
import re
import zipfile
import warnings

import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

Location of of the imput files and location where output files will be stored

In [26]:
production_disp_path = Path("../../../data/raw/texas/pdq_lease_output/og_lease_cycle_disp.parquet")
lease_centroid_path = Path("../../../data/raw/texas/coordinates/lease_centroids.parquet")

output_folder = Path("../../../data/raw/texas/coordinates")
output_folder.mkdir(parents=True, exist_ok=True)

Loading files

In [27]:
prod_disp = pd.read_parquet(production_disp_path)
lease_centroid = pd.read_parquet(lease_centroid_path)

print("Production shape:", prod_disp.shape)
print("Lease/well shape:", lease_centroid.shape)

print("\nProduction columns:")
print(prod_disp.columns.tolist())

print("\nLease/well columns:")
print(lease_centroid.columns.tolist())

Production shape: (16761949, 27)
Lease/well shape: (169051, 8)

Production columns:
['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'CYCLE_YEAR_MONTH', 'FIELD_NO', 'oil_pipeline_bbl', 'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl', 'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl', 'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl', 'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf', 'csgd_transmission_mcf', 'csgd_processing_plant_mcf', 'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf', 'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf', 'csgd_no_disp_code_mcf', 'oil_sold_total_bbl', 'total_vented_flared_mcf']

Lease/well columns:
['lease_key', 'n_wells_with_coordinates', 'county_name', 'oil_gas_code_norm', 'district_no_norm', 'lease_no_norm', 'lease_longitude', 'lease_latitude']


In [28]:
prod_disp.columns = prod_disp.columns.str.lower()
lease_centroid.lease_key.head(5)

0    O_01_00002
1    O_01_00003
2    O_01_00005
3    O_01_00006
4    O_01_00013
Name: lease_key, dtype: string

Production file is lease-level and lease/well file links leases to API numbers. We need a stable lease key to identify them between files. We do oil/gas code + district code + lease code.

In [29]:
def clean_text_series(s):
    return (
        s.astype("string")
         .str.strip()
         .str.upper()
         .str.replace(r"\.0$", "", regex=True)
    )

def normalize_district(s):
    s = clean_text_series(s)
    
    # RRC districts can be 01, 02, 03, 04, 05, 06, 7B, 7C, 08, 8A, 09, 10, etc.
    # If purely numeric, pad to 2 digits. Leave 8A, 7B, etc. as-is.
    return s.apply(lambda x: x.zfill(2) if pd.notna(x) and x.isdigit() else x)

def normalize_lease_no(s):
    s = clean_text_series(s)
    
    # Many RRC lease numbers are 5 digits.
    # If your source uses a different convention, inspect before changing.
    return s.apply(lambda x: x.zfill(5) if pd.notna(x) and x.isdigit() else x)

def normalize_oil_gas_code(s):
    return clean_text_series(s)

In [30]:
for df in [prod_disp]:
    df["oil_gas_code_norm"] = normalize_oil_gas_code(df["oil_gas_code"])
    df["district_no_norm"] = normalize_district(df["district_no"])
    df["lease_no_norm"] = normalize_lease_no(df["lease_no"])

    df["lease_key"] = (
        df["oil_gas_code_norm"] + "_" +
        df["district_no_norm"] + "_" +
        df["lease_no_norm"]
    )

In [31]:
prod_disp[['oil_gas_code','district_no','lease_no','lease_key']].head()

,oil_gas_code,district_no,lease_no,lease_key
0,O,08,09073,O_08_09073
1,O,10,27510,O_10_27510
2,O,10,27510,O_10_27510
3,O,10,27510,O_10_27510
4,O,10,27541,O_10_27541


In [32]:
prod_disp_coords = prod_disp.merge(
    lease_centroid[
        [
            "lease_key",
            "n_wells_with_coordinates",
            "county_name",
            "lease_longitude",
            "lease_latitude"
        ]
    ],
    on="lease_key",
    how="left"
)
prod_disp_coords = prod_disp_coords[
    prod_disp_coords["lease_longitude"].notna() &
    prod_disp_coords["lease_latitude"].notna()
]

Making this into a geo dataframe

In [35]:
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(prod_disp_coords['lease_longitude'],prod_disp_coords['lease_latitude'])]

In [36]:


prod_disp_coords = gpd.GeoDataFrame(
    prod_disp_coords,
    geometry= geometry,
    crs="EPSG:4326"
)

print("Production rows:", len(prod_disp_coords))
print(f"Production rows with lease centroid: {prod_disp_coords['geometry'].notna().mean():.2%}")

prod_disp_coords.head()

Production rows: 16658071
Production rows with lease centroid: 100.00%


,oil_gas_code,district_no,lease_no,cycle_year_month,field_no,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,oil_tank_cleaning_bbl,oil_circulating_bbl,...,total_vented_flared_mcf,oil_gas_code_norm,district_no_norm,lease_no_norm,lease_key,n_wells_with_coordinates,county_name,lease_longitude,lease_latitude,geometry
0,O,08,09073,202603,89812001,0,165,0,0,0,...,0,O,08,09073,O_08_09073,17.0,THROCKMORTON,-99.195238,33.016732,POINT (-99.19524 33.01673)
1,O,10,27510,202601,19541001,0,146,0,0,0,...,0,O,10,27510,O_10_27510,4.0,MITCHELL,-101.055068,32.487597,POINT (-101.05507 32.4876)
2,O,10,27510,202602,19541001,0,154,0,0,0,...,0,O,10,27510,O_10_27510,4.0,MITCHELL,-101.055068,32.487597,POINT (-101.05507 32.4876)
3,O,10,27510,202603,19541001,0,466,0,0,0,...,0,O,10,27510,O_10_27510,4.0,MITCHELL,-101.055068,32.487597,POINT (-101.05507 32.4876)
4,O,10,27541,202601,19541001,0,5,0,0,0,...,0,O,10,27541,O_10_27541,6.0,MITCHELL,-101.048186,32.485134,POINT (-101.04819 32.48513)


In [37]:
prod_disp_coords.drop(columns="geometry").to_parquet(
    output_folder / "production_disp_with_lease_coordinates.parquet",
    index=False
)

prod_disp_coords.to_parquet(
    output_folder / "production_disp_with_lease_coordinates.geoparquet"
)

In [38]:
prod_disp_coords.total_bounds

array([-104.83703932,   26.05147348,  -93.54891193,   36.49897029])

Checking if any leases did not get coordinates

In [39]:
prod_leases = prod_disp[
    ["lease_key", "oil_gas_code", "district_no", "lease_no"]
].drop_duplicates()

lease_coord_check = prod_leases.merge(
    lease_centroid[["lease_key"]].assign(has_coordinates=True),
    on="lease_key",
    how="left"
)

# Important: force real boolean dtype
lease_coord_check["has_coordinates"] = (
    lease_coord_check["has_coordinates"]
    .fillna(False)
    .astype(bool)
)

print(f"Production leases with coordinates: {lease_coord_check['has_coordinates'].mean():.2%}")

Production leases with coordinates: 99.48%


In [40]:
missing_prod_leases = lease_coord_check.loc[
    ~lease_coord_check["has_coordinates"]
].copy()

missing_prod_leases.shape

(708, 5)

<h3>Map check <h3>

In [41]:
prod_disp_coords.dropna(subset=["geometry"]).sample(
    min(5000, prod_disp_coords["geometry"].notna().sum()),
    random_state=42
).explore(
    tiles="CartoDB positron",
    tooltip=[
        "lease_key",
        "cycle_year_month",
        "n_wells_with_coordinates"
    ]
)